# KaBuM — Sistema de Recomendação de PC
Duas modalidades:
- **Modo 1:** Informa o orçamento → recebe uma build completa sugerida
- **Modo 2:** Escolhe uma peça → recebe sugestões de peças compatíveis

In [1]:
import re
import pathlib
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Carregar dados com features

In [2]:
DATA_DIR    = pathlib.Path(r"C:\Users\julia\OneDrive\Área de Trabalho\Projetos\Perspectivas de Dados\perspectiva-dado\Projetos\Projeto 1\00_Dados")
DATA_COLETA = "2026-05-16"  # ajuste para a data da sua coleta

pasta = DATA_DIR / DATA_COLETA
categorias = ["cpu", "gpu", "placa_mae", "ssd", "fonte", "ram"]

dfs = {}
for cat in categorias:
    arquivo = pasta / f"kabum_{cat}_{DATA_COLETA}_features.csv"
    dfs[cat] = pd.read_csv(arquivo, encoding="utf-8-sig")
    print(f"{cat:10s}: {len(dfs[cat])} produtos")

cpu       : 482 produtos
gpu       : 629 produtos
placa_mae : 842 produtos
ssd       : 867 produtos
fonte     : 616 produtos
ram       : 1200 produtos


## 2. Pré-processamento
Filtra produtos disponíveis e com preço válido.

In [3]:
def preparar(df):
    df = df[df["disponivel"] == True].copy()
    df = df[df["preco_pix"] > 0].copy()
    df = df.reset_index(drop=True)
    return df

for cat in categorias:
    antes = len(dfs[cat])
    dfs[cat] = preparar(dfs[cat])
    print(f"{cat:10s}: {antes} → {len(dfs[cat])} produtos disponíveis")

cpu       : 482 → 482 produtos disponíveis
gpu       : 629 → 629 produtos disponíveis
placa_mae : 842 → 842 produtos disponíveis
ssd       : 867 → 867 produtos disponíveis
fonte     : 616 → 616 produtos disponíveis
ram       : 1200 → 1200 produtos disponíveis


## 3. Score de custo-benefício
Combina preço normalizado e avaliação para rankear produtos dentro da mesma categoria.

In [29]:
def calcular_score(df, peso_preco=0.6, peso_avaliacao=0.4):
    df = df.copy()
    # Penaliza produtos sem avaliações (provavelmente irrelevantes)
    df["avaliacao"] = df["avaliacao"].fillna(0)
    df["num_avaliacoes"] = df["num_avaliacoes"].fillna(0)

    p_min, p_max = df["preco_pix"].min(), df["preco_pix"].max()
    df["score_preco"] = 1 - (df["preco_pix"] - p_min) / (p_max - p_min) if p_max > p_min else 1.0

    a_min, a_max = df["avaliacao"].min(), df["avaliacao"].max()
    df["score_aval"] = (df["avaliacao"] - a_min) / (a_max - a_min) if a_max > a_min else 0.5

    # Boost para produtos com muitas avaliações (mais confiáveis)
    df["score_popularidade"] = df["num_avaliacoes"].clip(upper=500) / 500

    df["score"] = (
        0.5 * df["score_preco"] +
        0.3 * df["score_aval"] +
        0.2 * df["score_popularidade"]
    )
    return df

for cat in categorias:
    dfs[cat] = calcular_score(dfs[cat])

print("Scores calculados para todas as categorias.")


# Palavras-chave obrigatórias por categoria (produto precisa conter pelo menos uma)
FILTROS_CATEGORIA = {
    "cpu":       r"processador|ryzen|core i[3579]|core ultra|athlon|xeon",
    "gpu":       r"(placa de v[ií]deo|geforce|radeon).*(rtx|gtx|\brx\b|arc)|rtx\s?\d{4}|gtx\s?\d{4}|\brx\s?\d{4}",
    "placa_mae": r"placa.m[ãa]e|motherboard",
    "ram":       r"mem[oó]ria ram|mem[oó]ria ddr",
    "ssd":       r"\bssd\b.*(gb|tb)|(gb|tb).*\bssd\b|nvme.*(gb|tb)|(gb|tb).*nvme",
    "fonte":     r"fonte de alimenta|fonte atx|fonte.*[0-9]+w",
    
}

def filtrar_categoria(df, cat):
    padrao = FILTROS_CATEGORIA.get(cat)
    if not padrao:
        return df
    mask = df["nome"].str.contains(padrao, case=False, na=False)
    df = df[mask]
    # Remove RAM de notebook do pool desktop
    if cat == "ram":
        df = df[~df["nome"].str.contains(r"notebook|sodimm|so-dimm", case=False, na=False)]
    return df

Scores calculados para todas as categorias.


## 4. Regras de compatibilidade

In [5]:
def compativel_cpu_mobo(cpu, mobo):
    """CPU e placa-mãe precisam ter o mesmo socket."""
    s_cpu  = cpu.get("cpu_socket")
    s_mobo = mobo.get("mobo_socket")
    if pd.isna(s_cpu) or pd.isna(s_mobo):
        return True  # sem info suficiente, não bloqueia
    return str(s_cpu).upper() == str(s_mobo).upper()


def compativel_ram_mobo(ram, mobo):
    """RAM precisa ter a mesma geração DDR da placa-mãe."""
    ddr_ram  = ram.get("ram_geracao")
    ddr_mobo = mobo.get("mobo_ddr")
    if pd.isna(ddr_ram) or pd.isna(ddr_mobo):
        return True
    return str(ddr_ram).upper() == str(ddr_mobo).upper()


def compativel_ram_cpu(ram, cpu):
    """RAM precisa ser suportada pela CPU (via tabela de referência de socket)."""
    ddr_ram = ram.get("ram_geracao")
    ddr_cpu = cpu.get("cpu_ddr_suportado")
    if pd.isna(ddr_ram) or pd.isna(ddr_cpu):
        return True
    # cpu_ddr_suportado pode ser "DDR4/DDR5" (LGA1700)
    return str(ddr_ram).upper() in str(ddr_cpu).upper()


def compativel_fonte_componentes(fonte, cpu, gpu):
    """Fonte precisa ter wattagem >= (TDP_CPU + TDP_GPU) * 1.25."""
    w_fonte  = fonte.get("fonte_wattagem")
    tdp_cpu  = cpu.get("cpu_tdp_w")
    tdp_gpu  = gpu.get("gpu_tdp_w") if gpu is not None else 0
    if pd.isna(w_fonte):
        return True
    tdp_total = (tdp_cpu if pd.notna(tdp_cpu) else 65) + (tdp_gpu if pd.notna(tdp_gpu) else 150)
    return w_fonte >= tdp_total * 1.25


def compativel_ssd_mobo(ssd, mobo):
    """SSD NVMe precisa de slot M.2 disponível na placa-mãe."""
    interface = ssd.get("ssd_interface")
    slots_m2  = mobo.get("mobo_slots_m2")
    if pd.isna(interface) or interface == "SATA":
        return True  # SATA sempre compatível
    if pd.isna(slots_m2):
        return True  # sem info, não bloqueia
    return int(slots_m2) > 0


print("Regras de compatibilidade definidas.")

Regras de compatibilidade definidas.


## 5. Modo 1 — Build por orçamento
Dado um orçamento total, sugere a melhor combinação de peças compatíveis.

In [33]:
def calcular_score_build(df, orcamento_cat):
    """
    Score relativo ao orçamento disponível para a categoria.
    Prioriza o melhor produto que cabe, não o mais barato do dataset.
    """
    df = df.copy()
    df_validos = df[df["preco_pix"] <= orcamento_cat].copy() 
    if df_validos.empty:
        df_validos = df.sort_values("preco_pix").head(5).copy() 

    p_min, p_max = df_validos["preco_pix"].min(), df_validos["preco_pix"].max()
    df_validos["score_preco"] = (
        (df_validos["preco_pix"] - p_min) / (p_max - p_min)  # maior preço = maior score
        if p_max > p_min else 0.5
    )

    a_min, a_max = df_validos["avaliacao"].min(), df_validos["avaliacao"].max()
    df_validos["score_aval"] = (
        (df_validos["avaliacao"].fillna(0) - a_min) / (a_max - a_min)
        if a_max > a_min else 0.5
    )

    df_validos["score_pop"] = df_validos["num_avaliacoes"].fillna(0).clip(upper=500) / 500

    df_validos["score"] = (
        0.4 * df_validos["score_preco"] +   # maior preço = melhor (dentro do limite)
        0.4 * df_validos["score_aval"] +
        0.2 * df_validos["score_pop"]
    )
    return df_validos

def sugerir_build(orcamento, incluir_gpu=True, top_candidatos=5):
    pesos = {
        "cpu":       0.22,
        "gpu":       0.35 if incluir_gpu else 0.0,
        "placa_mae": 0.15,
        "ram":       0.10,
        "ssd":       0.08,
        "fonte":     0.10,
    }
    if not incluir_gpu:
        total = sum(v for k, v in pesos.items() if k != "gpu")
        pesos = {k: v/total for k, v in pesos.items() if k != "gpu"}

    limites = {cat: orcamento * peso for cat, peso in pesos.items()}

    # Score relativo ao limite de cada categoria
    candidatos = {}
    for cat, limite in limites.items():
        df_scored = calcular_score_build(dfs[cat], limite)
        candidatos[cat] = df_scored.sort_values("score", ascending=False).head(top_candidatos)

    melhor_build = None
    melhor_score = -1

    cats = ["cpu", "placa_mae", "ram", "ssd", "fonte"] + (["gpu"] if incluir_gpu else [])
    for _, cpu in candidatos["cpu"].iterrows():
        for _, mobo in candidatos["placa_mae"].iterrows():
            if not compativel_cpu_mobo(cpu, mobo): continue
            for _, ram in candidatos["ram"].iterrows():
                if not compativel_ram_mobo(ram, mobo): continue
                if not compativel_ram_cpu(ram, cpu): continue
                for _, ssd in candidatos["ssd"].iterrows():
                    if not compativel_ssd_mobo(ssd, mobo): continue
                    gpu_rows = candidatos["gpu"].iterrows() if incluir_gpu else [(None, None)]
                    for _, gpu in gpu_rows:
                        for _, fonte in candidatos["fonte"].iterrows():
                            if not compativel_fonte_componentes(fonte, cpu, gpu): continue

                            pecas = {"cpu": cpu, "placa_mae": mobo, "ram": ram,
                                     "ssd": ssd, "fonte": fonte}
                            if incluir_gpu and gpu is not None:
                                pecas["gpu"] = gpu

                            preco_total = sum(p["preco_pix"] for p in pecas.values())
                            if preco_total > orcamento: continue

                            score_total = sum(p["score"] for p in pecas.values())
                            if score_total > melhor_score:
                                melhor_score = score_total
                                melhor_build = pecas

    return melhor_build


def exibir_build(build):
    if build is None:
        print("Nenhuma build compatível encontrada para este orçamento.")
        return

    linhas = []
    total = 0
    for cat, peca in build.items():
        preco = peca["preco_pix"]
        total += preco
        linhas.append({
            "Categoria":  cat.upper(),
            "Produto":    peca["nome"][:70],
            "Preço (R$)": preco,
            "Avaliação":  peca.get("avaliacao", "-"),
            "Score":      round(peca["score"], 3),
            "URL":        peca.get("url", ""),
        })

    df_build = pd.DataFrame(linhas)
    display(df_build[["Categoria", "Produto", "Preço (R$)", "Avaliação", "Score"]])
    print(f"\nTotal: R$ {total:,.2f}")
    print("\nLinks:")
    for l in linhas:
        print(f"  {l['Categoria']}: {l['URL']}")

In [38]:
# ─── CONFIGURE AQUI ───
ORCAMENTO   = 20000
INCLUIR_GPU = True
# ──────────────────────

print(f"Buscando melhor build até R$ {ORCAMENTO:,.2f}...\n")
build = sugerir_build(ORCAMENTO, incluir_gpu=INCLUIR_GPU)
exibir_build(build)

Buscando melhor build até R$ 20,000.00...



,Categoria,Produto,Preço (R$),Avaliação,Score
0,CPU,"Processador AMD Ryzen 7 5700G, 3.8GHz (4.6GHz Max Turbo)...",2356.97,5,0.83
1,PLACA_MAE,"Placa-Mãe MSI MPG B550 Gaming Plus, AMD AM4, ATX, DDR4, ...",1199.99,5,0.76
2,RAM,"Memória RAM Kingston Fury Beast, 16GB, 3200MHz, DDR4, CL...",1121.71,5,0.82
3,SSD,"SSD Kingston NV3, 1 TB, M.2 2280, PCIe 4.0 x4, NVMe, Lei...",1099.99,5,0.87
4,FONTE,"Fonte Corsair HX1000i, 1000W, 80 Plus Platinum, Modular,...",1999.99,5,0.80
5,GPU,Placa de Vídeo Gigabyte RTX 5070 EAGLE OC SFF 12G NVIDIA...,6999.99,5,0.82



Total: R$ 14,778.64

Links:
  CPU: https://www.kabum.com.br/produto/181089/processador-amd-ryzen-7-5700g-3-8ghz-4-6ghz-max-turbo-cache-20mb-8-nucleos-16-threads-video-integrado-am4-100-100000263box
  PLACA_MAE: https://www.kabum.com.br/produto/114335/placa-mae-msi-mpg-b550-gaming-plus-amd-am4-atx-ddr4-preto-mpg-b550-gaming-plus
  RAM: https://www.kabum.com.br/produto/172366/memoria-ram-kingston-fury-beast-16gb-3200mhz-ddr4-cl16-preto-kf432c16bb1-16
  SSD: https://www.kabum.com.br/produto/621162/ssd-kingston-nv3-1-tb-m-2-2280-pcie-4-0-x4-nvme-leitura-6000-mb-s-gravacao-4000-mb-s-azul-snv3s-1000g
  FONTE: https://www.kabum.com.br/produto/447386/fonte-corsair-hx1000i-1000w-80-plus-platinum-modular-com-cabo-preto-cp-9020259-ww
  GPU: https://www.kabum.com.br/produto/714573/placa-de-video-gigabyte-rtx-5070-eagle-oc-sff-12g-nvidia-geforce-12gb-gddr7-192bits-rgb-dlss-ray-tracing-9vn5070eo-00-g10


## 6. Modo 2 — Peças compatíveis com uma peça escolhida
Dado um produto específico, retorna as melhores peças compatíveis de outra categoria.

In [20]:
def buscar_peca(categoria, termo):
    """
    Busca uma peça pelo nome (busca parcial, case-insensitive).
    Retorna o produto com maior score entre os encontrados.
    """
    df = dfs[categoria]
    resultado = df[df["nome"].str.contains(termo, case=False, na=False)]
    if resultado.empty:
        print(f"Nenhum produto encontrado em '{categoria}' com o termo '{termo}'.")
        return None
    return resultado.sort_values("score", ascending=False).iloc[0]


def recomendar_compativeis(peca_ref, cat_ref, cat_alvo, orcamento_max=None, top_n=10):
    """
    Dado um produto de referência, retorna os melhores produtos compatíveis
    de outra categoria, ordenados por score.

    Parâmetros:
        peca_ref    : Series — produto escolhido pelo usuário
        cat_ref     : str — categoria da peça de referência (ex: 'cpu')
        cat_alvo    : str — categoria que quer receber sugestões (ex: 'placa_mae')
        orcamento_max: float — limite de preço opcional
        top_n       : int — quantas sugestões retornar
    """
    df_alvo = dfs[cat_alvo].copy()

    if orcamento_max:
        df_alvo = df_alvo[df_alvo["preco_pix"] <= orcamento_max]

    # Aplica regra de compatibilidade correta para o par de categorias
    par = tuple(sorted([cat_ref, cat_alvo]))

    def checar(row):
        if par == tuple(sorted(["cpu", "placa_mae"])):
            cpu  = peca_ref if cat_ref == "cpu"  else row
            mobo = peca_ref if cat_ref == "placa_mae" else row
            return compativel_cpu_mobo(cpu, mobo)
        if par == tuple(sorted(["ram", "placa_mae"])):
            ram  = peca_ref if cat_ref == "ram"  else row
            mobo = peca_ref if cat_ref == "placa_mae" else row
            return compativel_ram_mobo(ram, mobo)
        if par == tuple(sorted(["ram", "cpu"])):
            ram = peca_ref if cat_ref == "ram" else row
            cpu = peca_ref if cat_ref == "cpu" else row
            return compativel_ram_cpu(ram, cpu)
        if par == tuple(sorted(["ssd", "placa_mae"])):
            ssd  = peca_ref if cat_ref == "ssd"  else row
            mobo = peca_ref if cat_ref == "placa_mae" else row
            return compativel_ssd_mobo(ssd, mobo)
        if par == tuple(sorted(["fonte", "cpu"])) or par == tuple(sorted(["fonte", "gpu"])):
            cpu  = peca_ref if cat_ref == "cpu"   else row if cat_alvo == "cpu"  else pd.Series({"cpu_tdp_w": None})
            gpu  = peca_ref if cat_ref == "gpu"   else row if cat_alvo == "gpu"  else None
            fonte = peca_ref if cat_ref == "fonte" else row
            return compativel_fonte_componentes(fonte, cpu, gpu)
        return True  # par sem regra específica — retorna todos

    mask = df_alvo.apply(checar, axis=1)
    compativeis = df_alvo[mask].sort_values("score", ascending=False).head(top_n)

    return compativeis


def exibir_compativeis(df_result, cat_alvo):
    if df_result.empty:
        print("Nenhuma peça compatível encontrada.")
        return
    cols_base = ["nome", "preco_pix", "avaliacao", "score", "fabricante"]
    cols_extra = {
        "cpu":       ["cpu_socket", "cpu_marca", "cpu_ddr_suportado"],
        "placa_mae": ["mobo_socket", "mobo_chipset", "mobo_ddr"],
        "ram":       ["ram_geracao", "ram_gb", "ram_mhz"],
        "gpu":       ["gpu_modelo", "gpu_vram_gb", "gpu_tdp_w"],
        "ssd":       ["ssd_interface", "ssd_capacidade_gb"],
        "fonte":     ["fonte_wattagem", "fonte_certificacao"],
    }
    cols = cols_base + [c for c in cols_extra.get(cat_alvo, []) if c in df_result.columns]
    display(df_result[cols].reset_index(drop=True))

In [21]:
# ─── CONFIGURE AQUI ───
CATEGORIA_REF  = "cpu"         # categoria da peça que você já tem/escolheu
TERMO_BUSCA    = "Ryzen 5 5600"  # parte do nome do produto
CATEGORIA_ALVO = "placa_mae"   # categoria que quer ver sugestões
ORCAMENTO_MAX  = None          # ex: 1500.0 para limitar por preço, ou None
TOP_N          = 10            # quantas sugestões mostrar
# ──────────────────────

peca_ref = buscar_peca(CATEGORIA_REF, TERMO_BUSCA)

if peca_ref is not None:
    print(f"Peça de referência: {peca_ref['nome']}")
    print(f"Preço: R$ {peca_ref['preco_pix']:.2f} | Avaliação: {peca_ref.get('avaliacao', '-')}")
    print(f"\nTop {TOP_N} {CATEGORIA_ALVO.upper()} compatíveis:\n")

    resultado = recomendar_compativeis(
        peca_ref, CATEGORIA_REF, CATEGORIA_ALVO,
        orcamento_max=ORCAMENTO_MAX, top_n=TOP_N
    )
    exibir_compativeis(resultado, CATEGORIA_ALVO)

Peça de referência: Processador AMD Ryzen 5 5600GT, 3.6 GHz, (4.6GHz Max Turbo), Cache 4MB, 6 Núcleos, 12 Threads, AM4 - 100-100001488BOX
Preço: R$ 1099.98 | Avaliação: 5

Top 10 PLACA_MAE compatíveis:



,nome,preco_pix,avaliacao,score,fabricante,mobo_socket,mobo_chipset,mobo_ddr
0,"Placa-Mãe MSI A520M-A PRO, AMD AM4, mATX, DDR4, Preto - ...",439.99,5,0.99,MSI,AM4,NaN,DDR4
1,"Placa-Mãe ASUS TUF GAMING A520M-PLUS II, AMD AM4, mATX, ...",669.99,5,0.98,ASUS,AM4,NaN,DDR4
2,"Placa-Mãe ASRock B450M Steel Legend, AMD AM4, mATX, DDR4...",799.98,5,0.98,ASRock,AM4,NaN,DDR4
3,"Placa-Mãe MSI B550M Pro-VDH WiFi, AMD AM4, mATX, DDR4, P...",930.00,5,0.97,MSI,AM4,NaN,DDR4
4,"Placa-Mãe Gigabyte B550M Aorus Elite Rev. 1.3, AMD AM4, ...",950.64,5,0.97,Gigabyte,AM4,NaN,DDR4
5,"Placa-Mãe ASUS TUF Gaming B550M-Plus, AMD AM4, mATX, DDR...",999.99,5,0.97,ASUS,AM4,NaN,DDR4
6,"Placa-Mãe MSI MPG B550 Gaming Plus, AMD AM4, ATX, DDR4, ...",1199.99,5,0.96,MSI,AM4,B550,DDR4
7,"Placa-Mãe ASUS Prime A520M-E, AMD AM4, mATX, DDR4, Preto...",439.99,5,0.85,ASUS,AM4,NaN,DDR4
8,"Placa-Mãe Gigabyte A520M K V2 Rev. 1.0, AMD, Micro ATX, ...",479.99,5,0.85,Gigabyte,NaN,NaN,DDR4
9,"Placa-Mãe Asus TUF Gaming B550M-PLUS, AMD AM4, mATX , DD...",798.06,5,0.83,ASUS,AM4,NaN,DDR4
